In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

project_path = "/content/drive/MyDrive/AI Academic Assistant"

print(os.listdir(project_path))

['agents.py', 'app.py', 'syllabus']


In [ ]:
!pip install streamlit google-genai chromadb python-dotenv pypdf python-docx pillow requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 92.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 103.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 112.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 98.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 63.0 MB/s  0:00:00
  Attempting uninstall: urllib3
    Found existing installation: urllib3 2.5.0
    Uninstalling urllib3-2.5.0:
      Successfully uninstalled urllib3-2.5.0
  Attempting uninstall: opentelemetry-api
    Found existing installation: opentelemetry-api 1.42.1
    Uninstalling opentelemetry-api-1.42.1:
      Successfully uninstalled opentelemetry-api-1.42.1
  Attempting uninstall: oauthlib
    Found existing installation: oauthlib 3.3.1
    Uninstalling oauthlib-3.3.1:
      Successfully uninstalled oauthlib-3.3.1
  Attempting uninstall: opentelemetry-semantic-conventions
    Found existing installation: opentelemetry

In [ ]:
app_path = "/content/drive/MyDrive/AI Academic Assistant/app.py"

with open(app_path, "r", encoding="utf-8") as file:
    print(file.read())

import os
import hashlib
from pathlib import Path

import streamlit as st
import chromadb
import pytesseract
from dotenv import load_dotenv
from google import genai
from google.genai import types
from pypdf import PdfReader
from docx import Document

from agents import QueryAgent, RetrievalAgent, AnswerAgent, EvaluationAgent

# --------------------------------------------------
# 1. PAGE CONFIGURATION
# --------------------------------------------------
st.set_page_config(
    page_title="AI-Powered Academic Knowledge Assistant",
    page_icon="🎓",
    layout="wide"
)

BASE_DIR = Path(__file__).resolve().parent
SYLLABUS_DIR = BASE_DIR / "syllabus"

pytesseract.pytesseract.tesseract_cmd = (
    r"C:\Program Files\Tesseract-OCR\tesseract.exe"
)
POPPLER_PATH = (
    r"C:\Users\kaviya kumar\Downloads\Release-26.09.0-0"
    r"\poppler-26.09.0\Library\bin"
)

# --------------------------------------------------
# 2. GEMINI CONFIGURATION
# --------------------------------------------------
lo

In [ ]:
agents_path = "/content/drive/MyDrive/AI Academic Assistant/agents.py"

with open(agents_path, "r", encoding="utf-8") as file:
    print(file.read())

class QueryAgent:
    def process_query(self, question):
        question = question.strip()
        return question if question else None


class RetrievalAgent:
    def retrieve_context(
        self, question, client, collection, subject_name, year, semester
    ):
        embedding_response = client.models.embed_content(
            model="gemini-embedding-001",
            contents=question
        )
        question_embedding = embedding_response.embeddings[0].values

        results = collection.query(
            query_embeddings=[question_embedding],
            n_results=5,
            where={
                "$and": [
                    {"subject": subject_name},
                    {"year": year},
                    {"semester": semester}
                ]
            }
        )

        documents = results.get("documents", [[]])[0]
        metadatas = results.get("metadatas", [[]])[0]
        sources = sorted({
            metadata["source"]
            for metadata in 

In [ ]:
from google.colab import userdata

NEXUS_API_KEY = userdata.get("NEXUS_API_KEY")

if NEXUS_API_KEY:
    print("Nexus API key loaded successfully!")
else:
    print("Nexus API key not found!")

Nexus API key loaded successfully!


In [ ]:
import requests

url = "https://nexusapi.navigatelabs.ai/v1/chat/completions"

headers = {
    "Authorization": f"Bearer {NEXUS_API_KEY}",
    "Content-Type": "application/json"
}

data = {
    "model": "nova-micro",
    "messages": [
        {"role": "user", "content": "Explain Artificial Intelligence in two simple sentences."}
    ]
}

response = requests.post(url, headers=headers, json=data)

print("Status Code:", response.status_code)

if response.status_code == 200:
    print(response.json()["choices"][0]["message"]["content"])
else:
    print(response.text)

Status Code: 200
Artificial Intelligence (AI) is a field of technology that focuses on creating machines capable of performing tasks that typically require human intelligence. AI systems use algorithms and data to make informed decisions and solve complex problems.


In [ ]:
import requests

response = requests.post(
    "https://nexusapi.navigatelabs.ai/v1/embeddings",
    headers={
        "Authorization": f"Bearer {NEXUS_API_KEY}",
        "Content-Type": "application/json"
    },
    json={
        "model": "text-embedding-3-small",
        "input": "Artificial Intelligence"
    }
)

print("Status Code:", response.status_code)
print(response.text[:500])

Status Code: 200
{"model":"text-embedding-3-small","data":[{"embedding":[-0.0172119140625,-0.007427215576171875,-0.020172119140625,-0.00971221923828125,0.01160430908203125,-0.006420135498046875,-0.0142669677734375,0.032318115234375,-0.00592803955078125,0.01541900634765625,0.0140228271484375,-0.06634521484375,-0.0225830078125,-0.043914794921875,-0.00807952880859375,-0.0156707763671875,0.00811767578125,-0.0266571044921875,0.033172607421875,-0.043487548828125,0.01325225830078125,0.00911712646484375,-0.0201873779296


In [ ]:
%%writefile /content/drive/MyDrive/AI Academic Assistant/agents.py

import requests


NEXUS_BASE_URL = "https://nexusapi.navigatelabs.ai/v1"


class QueryAgent:
    def process_query(self, question):
        question = question.strip()
        return question if question else None


class RetrievalAgent:
    def retrieve_context(
        self, question, client, collection, subject_name, year, semester
    ):
        response = requests.post(
            f"{NEXUS_BASE_URL}/embeddings",
            headers={
                "Authorization": f"Bearer {client}",
                "Content-Type": "application/json"
            },
            json={
                "model": "text-embedding-3-small",
                "input": question
            },
            timeout=60
        )

        response.raise_for_status()
        question_embedding = response.json()["data"][0]["embedding"]

        results = collection.query(
            query_embeddings=[question_embedding],
            n_results=5,
            where={
                "$and": [
                    {"subject": subject_name},
                    {"year": year},
                    {"semester": semester}
                ]
            }
        )

        documents = results.get("documents", [[]])[0]
        metadatas = results.get("metadatas", [[]])[0]

        sources = sorted({
            metadata["source"]
            for metadata in metadatas
            if metadata and metadata.get("source")
        })

        return documents, sources


class AnswerAgent:
    def __init__(self, client):
        self.client = client

    def generate_answer(self, question, context):
        if not context:
            return "This topic is not available in the selected subject syllabus."

        syllabus_context = "\n\n".join(context)

        prompt = f"""
You are an AI Academic Tutor.
Answer the student's question using the selected subject syllabus context.

Rules:
1. First check whether the topic is related to the selected syllabus context.
2. If the topic is unrelated to the selected subject, reply exactly:
   "This topic is not available in the selected subject syllabus."
3. If the syllabus contains only a heading or brief mention, explain that topic
   using standard academic knowledge, while staying within the subject.
4. Give a clear, student-friendly answer with suitable headings and examples.
5. Do not claim that information is directly quoted from the syllabus unless
   it appears in the supplied context.

Selected syllabus context:
{syllabus_context}

Student question:
{question}
"""

        response = requests.post(
            f"{NEXUS_BASE_URL}/chat/completions",
            headers={
                "Authorization": f"Bearer {self.client}",
                "Content-Type": "application/json"
            },
            json={
                "model": "nova-micro",
                "messages": [
                    {"role": "user", "content": prompt}
                ],
                "temperature": 0.3
            },
            timeout=120
        )

        response.raise_for_status()

        result = response.json()
        return (
            result["choices"][0]["message"]["content"]
            or "The answer could not be generated."
        )


class EvaluationAgent:
    def evaluate_answer(self, answer):
        if not answer or not answer.strip():
            return "The answer could not be generated."

        if "not available in the selected subject syllabus" in answer.lower():
            return "The answer is not available in the selected subject syllabus."

        return "Answer generated successfully."

UsageError: unrecognized arguments: Academic Assistant/agents.py


In [ ]:
%%writefile "/content/drive/MyDrive/AI Academic Assistant/agents.py"

Overwriting /content/drive/MyDrive/AI Academic Assistant/agents.py


In [ ]:
agents_path = "/content/drive/MyDrive/AI Academic Assistant/agents.py"

with open(agents_path, "r", encoding="utf-8") as file:
    print(file.read()[:1000])

In [ ]:
import os

agents_path = "/content/drive/MyDrive/AI Academic Assistant/agents.py"

print("File exists:", os.path.exists(agents_path))
print("File size:", os.path.getsize(agents_path) if os.path.exists(agents_path) else 0)

File exists: True
File size: 1


In [ ]:
%%writefile "/content/drive/MyDrive/AI Academic Assistant/agents.py"

Overwriting /content/drive/MyDrive/AI Academic Assistant/agents.py


In [ ]:
%%writefile "/content/drive/MyDrive/AI Academic Assistant/agents.py"

import requests

NEXUS_BASE_URL = "https://nexusapi.navigatelabs.ai/v1"


class QueryAgent:
    def process_query(self, question):
        question = question.strip()
        return question if question else None


class RetrievalAgent:
    def retrieve_context(
        self, question, client, collection, subject_name, year, semester
    ):
        response = requests.post(
            f"{NEXUS_BASE_URL}/embeddings",
            headers={
                "Authorization": f"Bearer {client}",
                "Content-Type": "application/json"
            },
            json={
                "model": "text-embedding-3-small",
                "input": question
            },
            timeout=60
        )

        response.raise_for_status()
        question_embedding = response.json()["data"][0]["embedding"]

        results = collection.query(
            query_embeddings=[question_embedding],
            n_results=5,
            where={
                "$and": [
                    {"subject": subject_name},
                    {"year": year},
                    {"semester": semester}
                ]
            }
        )

        documents = results.get("documents", [[]])[0]
        metadatas = results.get("metadatas", [[]])[0]

        sources = sorted({
            metadata["source"]
            for metadata in metadatas
            if metadata and metadata.get("source")
        })

        return documents, sources


class AnswerAgent:
    def __init__(self, client):
        self.client = client

    def generate_answer(self, question, context):
        if not context:
            return "This topic is not available in the selected subject syllabus."

        syllabus_context = "\n\n".join(context)

        prompt = f"""
You are an AI Academic Tutor.
Answer the student's question using the selected subject syllabus context.

Rules:
1. First check whether the topic is related to the selected syllabus context.
2. If the topic is unrelated to the selected subject, reply exactly:
   "This topic is not available in the selected subject syllabus."
3. If the syllabus contains only a heading or brief mention, explain that topic
   using standard academic knowledge, while staying within the subject.
4. Give a clear, student-friendly answer with suitable headings and examples.
5. Do not claim that information is directly quoted from the syllabus unless
   it appears in the supplied context.

Selected syllabus context:
{syllabus_context}

Student question:
{question}
"""

        response = requests.post(
            f"{NEXUS_BASE_URL}/chat/completions",
            headers={
                "Authorization": f"Bearer {self.client}",
                "Content-Type": "application/json"
            },
            json={
                "model": "nova-micro",
                "messages": [
                    {"role": "user", "content": prompt}
                ],
                "temperature": 0.3
            },
            timeout=120
        )

        response.raise_for_status()
        result = response.json()

        return (
            result["choices"][0]["message"]["content"]
            or "The answer could not be generated."
        )


class EvaluationAgent:
    def evaluate_answer(self, answer):
        if not answer or not answer.strip():
            return "The answer could not be generated."

        if "not available in the selected subject syllabus" in answer.lower():
            return "The answer is not available in the selected subject syllabus."

        return "Answer generated successfully."

Overwriting /content/drive/MyDrive/AI Academic Assistant/agents.py


In [ ]:
import os

path = "/content/drive/MyDrive/AI Academic Assistant/agents.py"

print("File size:", os.path.getsize(path), "bytes")

with open(path, "r", encoding="utf-8") as f:
    print(f.read()[:300])

File size: 3577 bytes

import requests

NEXUS_BASE_URL = "https://nexusapi.navigatelabs.ai/v1"


class QueryAgent:
    def process_query(self, question):
        question = question.strip()
        return question if question else None


class RetrievalAgent:
    def retrieve_context(
        self, question, client, coll


In [ ]:
import ast

path = "/content/drive/MyDrive/AI Academic Assistant/agents.py"

with open(path, "r", encoding="utf-8") as f:
    code = f.read()

ast.parse(code)
print("agents.py syntax check successful!")

agents.py syntax check successful!


In [ ]:
app_path = "/content/drive/MyDrive/AI Academic Assistant/app.py"

with open(app_path, "r", encoding="utf-8") as file:
    app_code = file.read()

print(app_code[:5000])

import os
import hashlib
from pathlib import Path

import streamlit as st
import chromadb
import pytesseract
from dotenv import load_dotenv
from google import genai
from google.genai import types
from pypdf import PdfReader
from docx import Document

from agents import QueryAgent, RetrievalAgent, AnswerAgent, EvaluationAgent

# --------------------------------------------------
# 1. PAGE CONFIGURATION
# --------------------------------------------------
st.set_page_config(
    page_title="AI-Powered Academic Knowledge Assistant",
    page_icon="🎓",
    layout="wide"
)

BASE_DIR = Path(__file__).resolve().parent
SYLLABUS_DIR = BASE_DIR / "syllabus"

pytesseract.pytesseract.tesseract_cmd = (
    r"C:\Program Files\Tesseract-OCR\tesseract.exe"
)
POPPLER_PATH = (
    r"C:\Users\kaviya kumar\Downloads\Release-26.09.0-0"
    r"\poppler-26.09.0\Library\bin"
)

# --------------------------------------------------
# 2. GEMINI CONFIGURATION
# --------------------------------------------------
lo

In [ ]:
print(app_code[5000:])

---------------
def create_chunks(text, chunk_size=800, overlap=100):
    chunks = []
    start = 0
    while start < len(text):
        chunk = text[start:start + chunk_size]
        if chunk.strip():
            chunks.append(chunk)
        start += chunk_size - overlap
    return chunks

def extract_file_text(file_path):
    text = ""
    extension = file_path.suffix.lower()

    if extension == ".pdf":
        reader = PdfReader(str(file_path))
        for page_number, page in enumerate(reader.pages, start=1):
            page_text = page.extract_text() or ""
            if page_text.strip():
                text += page_text + "\n"
            else:
                from pdf2image import convert_from_path
                images = convert_from_path(
                    str(file_path),
                    first_page=page_number,
                    last_page=page_number,
                    poppler_path=POPPLER_PATH
                )
                for image in images:
             

In [ ]:
!apt-get update -qq
!apt-get install -y -qq tesseract-ocr poppler-utils
!pip install -q pytesseract pdf2image requests

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package poppler-utils.
(Reading database ... 122813 files and directories currently installed.)
Preparing to unpack .../poppler-utils_24.02.0-1ubuntu9.11_amd64.deb ...
Unpacking poppler-utils (24.02.0-1ubuntu9.11) ...
Setting up poppler-utils (24.02.0-1ubuntu9.11) ...
Processing triggers for man-db (2.12.0-4build2) ...


In [ ]:
import pytesseract
import subprocess

print("Tesseract version:")
print(subprocess.check_output(["tesseract", "--version"]).decode().split("\n")[0])

print("OCR package loaded successfully!")

Tesseract version:
tesseract 5.3.4
OCR package loaded successfully!


In [ ]:
import shutil

app_path = "/content/drive/MyDrive/AI Academic Assistant/app.py"
backup_path = "/content/drive/MyDrive/AI Academic Assistant/app_backup.py"

shutil.copy2(app_path, backup_path)

print("app.py backup created successfully!")

app.py backup created successfully!


In [ ]:
app_path = "/content/drive/MyDrive/AI Academic Assistant/app.py"

with open(app_path, "r", encoding="utf-8") as file:
    lines = file.readlines()

for i, line in enumerate(lines[:60], start=1):
    print(f"{i}: {line}", end="")

1: import os
2: import hashlib
3: from pathlib import Path
4: 
5: import streamlit as st
6: import chromadb
7: import pytesseract
8: from dotenv import load_dotenv
9: from google import genai
10: from google.genai import types
11: from pypdf import PdfReader
12: from docx import Document
13: 
14: from agents import QueryAgent, RetrievalAgent, AnswerAgent, EvaluationAgent
15: 
16: # --------------------------------------------------
17: # 1. PAGE CONFIGURATION
18: # --------------------------------------------------
19: st.set_page_config(
20:     page_title="AI-Powered Academic Knowledge Assistant",
21:     page_icon="🎓",
22:     layout="wide"
23: )
24: 
25: BASE_DIR = Path(__file__).resolve().parent
26: SYLLABUS_DIR = BASE_DIR / "syllabus"
27: 
28: pytesseract.pytesseract.tesseract_cmd = (
29:     r"C:\Program Files\Tesseract-OCR\tesseract.exe"
30: )
31: POPPLER_PATH = (
32:     r"C:\Users\kaviya kumar\Downloads\Release-26.09.0-0"
33:     r"\poppler-26.09.0\Library\bin"
34: )
35: 
36:

In [ ]:
app_path = "/content/drive/MyDrive/AI Academic Assistant/app.py"

with open(app_path, "r", encoding="utf-8") as file:
    code = file.read()

code = code.replace("from dotenv import load_dotenv\n", "")
code = code.replace("from google import genai\n", "")
code = code.replace("from google.genai import types\n", "")

start = code.index("# --------------------------------------------------\n# 2. GEMINI CONFIGURATION")
end = code.index("# --------------------------------------------------\n# 3. CHROMADB + AGENTS")

nexus_config = '''# --------------------------------------------------
# 2. NEXUS API CONFIGURATION
# --------------------------------------------------
from google.colab import userdata

NEXUS_API_KEY = userdata.get("NEXUS_API_KEY")

if not NEXUS_API_KEY:
    st.error("Nexus API key not found in Colab Secrets.")
    st.stop()

client = NEXUS_API_KEY

'''

code = code[:start] + nexus_config + code[end:]

with open(app_path, "w", encoding="utf-8") as file:
    file.write(code)

print("Nexus API configuration updated successfully!")

Nexus API configuration updated successfully!


In [ ]:
app_path = "/content/drive/MyDrive/AI Academic Assistant/app.py"

with open(app_path, "r", encoding="utf-8") as file:
    code = file.read()

old_paths = '''pytesseract.pytesseract.tesseract_cmd = (
    r"C:\\Program Files\\Tesseract-OCR\\tesseract.exe"
)
POPPLER_PATH = (
    r"C:\\Users\\kaviya kumar\\Downloads\\Release-26.09.0-0"
    r"\\poppler-26.09.0\\Library\\bin"
)'''

new_paths = '''# Colab/Linux OCR configuration
pytesseract.pytesseract.tesseract_cmd = "/usr/bin/tesseract"
POPPLER_PATH = None'''

if old_paths in code:
    code = code.replace(old_paths, new_paths)

    with open(app_path, "w", encoding="utf-8") as file:
        file.write(code)

    print("OCR paths updated successfully!")
else:
    print("Old OCR paths not found. No changes made.")

OCR paths updated successfully!


In [ ]:
app_path = "/content/drive/MyDrive/AI Academic Assistant/app.py"

with open(app_path, "r", encoding="utf-8") as file:
    code = file.read()

start = code.index('                response = client.models.embed_content(')
end = code.index('                ids, metadatas = [], []', start)

print(code[start:end])

                response = client.models.embed_content(
                    model="gemini-embedding-001",
                    contents=batch
                )
                embeddings = [item.values for item in response.embeddings]



In [ ]:
app_path = "/content/drive/MyDrive/AI Academic Assistant/app.py"

with open(app_path, "r", encoding="utf-8") as file:
    code = file.read()

old_code = '''                response = client.models.embed_content(
                    model="gemini-embedding-001",
                    contents=batch
                )
                embeddings = [item.values for item in response.embeddings]'''

new_code = '''                import requests

                response = requests.post(
                    "https://nexusapi.navigatelabs.ai/v1/embeddings",
                    headers={
                        "Authorization": f"Bearer {NEXUS_API_KEY}",
                        "Content-Type": "application/json"
                    },
                    json={
                        "model": "text-embedding-3-small",
                        "input": batch
                    },
                    timeout=120
                )

                response.raise_for_status()

                embedding_data = response.json()["data"]
                embedding_data.sort(key=lambda item: item["index"])
                embeddings = [
                    item["embedding"] for item in embedding_data
                ]'''

if old_code in code:
    code = code.replace(old_code, new_code)

    with open(app_path, "w", encoding="utf-8") as file:
        file.write(code)

    print("Nexus embedding integration updated successfully!")
else:
    print("Gemini embedding code not found. No changes made.")

Nexus embedding integration updated successfully!


In [ ]:
app_path = "/content/drive/MyDrive/AI Academic Assistant/app.py"

with open(app_path, "r", encoding="utf-8") as file:
    code = file.read()

for term in ["genai", "gemini", "generate_content", "embed_content"]:
    print(term, ":", term in code.lower())

genai : False
gemini : True
generate_content : False
embed_content : False


In [ ]:
for i, line in enumerate(code.splitlines(), start=1):
    if "gemini" in line.lower():
        print(f"Line {i}: {line}")

Line 333:             return "Gemini AI server is currently busy. Please try again later.", []
Line 339:             return "API access denied. Please check your Gemini API permissions.", []


In [ ]:
app_path = "/content/drive/MyDrive/AI Academic Assistant/app.py"

with open(app_path, "r", encoding="utf-8") as file:
    code = file.read()

code = code.replace(
    "Gemini AI server is currently busy. Please try again later.",
    "Nexus AI server is currently busy. Please try again later."
)

code = code.replace(
    "API access denied. Please check your Gemini API permissions.",
    "API access denied. Please check your Nexus API key and permissions."
)

with open(app_path, "w", encoding="utf-8") as file:
    file.write(code)

print("Gemini error messages changed to Nexus successfully!")

Gemini error messages changed to Nexus successfully!


In [ ]:
import ast

app_path = "/content/drive/MyDrive/AI Academic Assistant/app.py"

with open(app_path, "r", encoding="utf-8") as file:
    app_code = file.read()

ast.parse(app_code)

print("app.py syntax check successful!")

app.py syntax check successful!


In [ ]:
import chromadb

db_path = "/content/drive/MyDrive/AI Academic Knowledge/chroma_db"

chroma_client = chromadb.PersistentClient(path=db_path)

print("Collections:", chroma_client.list_collections())

Collections: []


In [ ]:
from pathlib import Path

syllabus_path = Path(
    "/content/drive/MyDrive/AI Academic Assistant/syllabus"
)

for year_folder in syllabus_path.iterdir():
    if year_folder.is_dir():
        print(f"\n{year_folder.name}:")
        files = list(year_folder.rglob("*"))

        for file in files:
            if file.is_file():
                print(" -", file.name)


2nd_year:

3rd_year:

4th_year:


In [ ]:
from pathlib import Path

syllabus_path = Path(
    "/content/drive/MyDrive/AI Academic Assistant/syllabus"
)

for year_folder in sorted(syllabus_path.iterdir()):
    if year_folder.is_dir():
        print(f"\n📁 {year_folder.name}")

        for semester_folder in sorted(year_folder.iterdir()):
            if semester_folder.is_dir():
                print(f"   📂 {semester_folder.name}")

                for file in semester_folder.iterdir():
                    if file.is_file():
                        print(f"      📄 {file.name}")


📁 2nd_year
   📂 semester_3
      📄 DBMS syllabus.pdf
      📄 DS syllabus.pdf
   📂 semester_4

📁 3rd_year
   📂 semester_5
      📄 AI syllabus.pdf
      📄 CD syllabus.pdf
      📄 DCN Syllabus.pdf
      📄 Electrical safety syllabus.pdf
      📄 MPMC syllabus.docx
   📂 semester_6

📁 4th_year
   📂 semester_7
      📄 Syllabus - FPPT.docx
      📄 cns syllabus.pdf
      📄 dl syllabus.pdf
      📄 ethics syllabus.pdf
   📂 semester_8


In [ ]:
!pip install -q streamlit


In [ ]:
print("Hello Bro!")


Hello Bro!


In [ ]:
!pip install -q streamlit

In [ ]:
import streamlit
print("Streamlit installed successfully!")
print("Version:", streamlit.__version__)

Streamlit installed successfully!
Version: 1.67.0


In [ ]:
import os
from google.colab import userdata

os.environ["NEXUS_API_KEY"] = userdata.get("NEXUS_API_KEY")

%cd "/content/drive/MyDrive/AI Academic Assistant"

!streamlit run app.py &>/content/streamlit.log &

/content/drive/MyDrive/AI Academic Assistant


In [ ]:
!cat /content/streamlit.log



2026-11-19 19:19:06.782 Uvicorn server started on :::8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.83.186.116:8501



In [ ]:
!npm install -g localtunnel

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏
added 22 packages in 3s
⠏
⠏3 packages are looking for funding
⠏  run `npm fund` for details
⠏

In [ ]:
!lt --port 8501

^C


In [ ]:
!lt --port 8501 --print-requests

^C


In [ ]:
!curl -I http://localhost:8501

curl: (7) Failed to connect to localhost port 8501 after 0 ms: Couldn't connect to server


In [ ]:
%cd "/content/drive/MyDrive/AI Academic Assistant"

!streamlit run app.py --server.port 8501

/content/drive/MyDrive/AI Academic Assistant


2026-11-19 19:29:39.704 Uvicorn server started on :::8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.83.186.116:8501

  Stopping...


In [ ]:
from google.colab import output
output.serve_kernel_port_as_window(8501)

Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>

In [ ]:
from google.colab import output
output.serve_kernel_port_as_iframe(8501)

<IPython.core.display.Javascript object>

In [ ]:
!curl -I http://localhost:8501

curl: (7) Failed to connect to localhost port 8501 after 0 ms: Couldn't connect to server


In [ ]:
!tail -40 /content/streamlit.log



2026-11-19 19:19:06.782 Uvicorn server started on :::8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.83.186.116:8501

  Stopping...


In [ ]:
import os
import subprocess
import time

os.chdir("/content/drive/MyDrive/AI Academic Assistant")

log_file = open("/content/streamlit.log", "w")

process = subprocess.Popen(
    ["streamlit", "run", "app.py", "--server.port", "8501"],
    stdout=log_file,
    stderr=subprocess.STDOUT
)

time.sleep(10)

print("Streamlit process running:", process.poll() is None)

Streamlit process running: True


In [ ]:
!curl -I http://localhost:8501

HTTP/1.1 200 OK
date: Thu, 19 Nov 2026 19:36:33 GMT
server: uvicorn
content-type: text/html; charset=utf-8
accept-ranges: bytes
content-length: 7863
last-modified: Thu, 19 Nov 2026 18:47:40 GMT
etag: "5afd62fef7275c36a1afa70de40032c5"
cache-control: no-cache



In [55]:
from google.colab import output
output.serve_kernel_port_as_iframe(8501)

<IPython.core.display.Javascript object>

In [56]:
import requests

response = requests.get("http://localhost:8501", timeout=10)

print("Status:", response.status_code)
print("Page title:", "AI Academic Assistant" in response.text)
print("Response size:", len(response.text))

Status: 200
Page title: False
Response size: 7863


In [57]:
print(open("/content/streamlit.log").read()[-5000:])

times.internal:8007
2026-11-19 19:39:58.680 Rejecting WebSocket connection with disallowed Origin or Host header: origin=https://8501-m-s-usw1c6-3w4cceqzgkuj1-c.us-west1.prod.r.colab.dev, host=m-s-usw1c6-3w4cceqzgkuj1.us-west1-c.c.codatalab-user-runtimes.internal:8007
2026-11-19 19:39:58.861 Rejecting WebSocket connection with disallowed Origin or Host header: origin=https://8501-m-s-usw1c6-3w4cceqzgkuj1-c.us-west1.prod.r.colab.dev, host=m-s-usw1c6-3w4cceqzgkuj1.us-west1-c.c.codatalab-user-runtimes.internal:8007
2026-11-19 19:40:03.968 Rejecting WebSocket connection with disallowed Origin or Host header: origin=https://8501-m-s-usw1c6-3w4cceqzgkuj1-c.us-west1.prod.r.colab.dev, host=m-s-usw1c6-3w4cceqzgkuj1.us-west1-c.c.codatalab-user-runtimes.internal:8007
2026-11-19 19:40:04.103 Rejecting WebSocket connection with disallowed Origin or Host header: origin=https://8501-m-s-usw1c6-3w4cceqzgkuj1-c.us-west1.prod.r.colab.dev, host=m-s-usw1c6-3w4cceqzgkuj1.us-west1-c.c.codatalab-user-runtime

In [58]:
process.terminate()
process.wait()

import subprocess
import time
import os

os.chdir("/content/drive/MyDrive/AI Academic Assistant")

log_file = open("/content/streamlit.log", "w")

process = subprocess.Popen(
    [
        "streamlit", "run", "app.py",
        "--server.port", "8501",
        "--server.address", "0.0.0.0",
        "--server.enableCORS", "false",
        "--server.enableXsrfProtection", "false",
        "--server.headless", "true"
    ],
    stdout=log_file,
    stderr=subprocess.STDOUT
)

time.sleep(10)

print("Server running:", process.poll() is None)

Server running: True


In [59]:
!curl -I http://localhost:8501

HTTP/1.1 200 OK
date: Thu, 19 Nov 2026 19:42:39 GMT
server: uvicorn
content-type: text/html; charset=utf-8
accept-ranges: bytes
content-length: 7863
last-modified: Thu, 19 Nov 2026 18:47:40 GMT
etag: "5afd62fef7275c36a1afa70de40032c5"
cache-control: no-cache



In [60]:
from google.colab import output
output.serve_kernel_port_as_iframe(8501)

<IPython.core.display.Javascript object>

In [63]:
# --------------------------------------------------
# 2. NEXUS API CONFIGURATION
# --------------------------------------------------

NEXUS_API_KEY = os.environ.get("NEXUS_API_KEY")

if not NEXUS_API_KEY:
    st.error("Nexus API key not found in environment variables.")
    st.stop()

client = NEXUS_API_KEY

In [64]:
print("Hello Bro! Colab is working!")

Hello Bro! Colab is working!


In [65]:
from google.colab import userdata
import os

key = userdata.get("NEXUS_API_KEY")

if key:
    os.environ["NEXUS_API_KEY"] = key
    print("Nexus API key loaded successfully!")
else:
    print("NEXUS_API_KEY not found in Colab Secrets.")

Nexus API key loaded successfully!


In [66]:
# Stop the previous Streamlit server
try:
    process.terminate()
    process.wait()
    print("Previous server stopped.")
except:
    print("No previous server found.")

# Start Streamlit with the Nexus API key
import os
import subprocess
import time

os.chdir("/content/drive/MyDrive/AI Academic Assistant")

log_file = open("/content/streamlit.log", "w")

process = subprocess.Popen(
    [
        "streamlit", "run", "app.py",
        "--server.port", "8501",
        "--server.address", "0.0.0.0",
        "--server.enableCORS", "false",
        "--server.enableXsrfProtection", "false",
        "--server.headless", "true"
    ],
    stdout=log_file,
    stderr=subprocess.STDOUT
)

time.sleep(10)

print("Streamlit server running:", process.poll() is None)

Previous server stopped.
Streamlit server running: True


In [67]:
from google.colab import output
output.serve_kernel_port_as_iframe(8501)

<IPython.core.display.Javascript object>

In [68]:
app_path = "/content/drive/MyDrive/AI Academic Assistant/app.py"

with open(app_path, "r") as f:
    lines = f.readlines()

for i in range(20, 42):
    print(f"{i+1}: {lines[i].rstrip()}")

21: 
22: BASE_DIR = Path(__file__).resolve().parent
23: SYLLABUS_DIR = BASE_DIR / "syllabus"
24: 
25: # Colab/Linux OCR configuration
26: pytesseract.pytesseract.tesseract_cmd = "/usr/bin/tesseract"
27: POPPLER_PATH = None
28: 
29: # --------------------------------------------------
30: # 2. NEXUS API CONFIGURATION
31: # --------------------------------------------------
32: from google.colab import userdata
33: 
34: NEXUS_API_KEY = userdata.get("NEXUS_API_KEY")
35: 
36: if not NEXUS_API_KEY:
37:     st.error("Nexus API key not found in Colab Secrets.")
38:     st.stop()
39: 
40: client = NEXUS_API_KEY
41: 
42: # --------------------------------------------------


In [69]:
app_path = "/content/drive/MyDrive/AI Academic Assistant/app.py"

with open(app_path, "r") as f:
    code = f.read()

old_code = '''from google.colab import userdata

NEXUS_API_KEY = userdata.get("NEXUS_API_KEY")'''

new_code = '''NEXUS_API_KEY = os.environ.get("NEXUS_API_KEY")'''

if old_code in code:
    code = code.replace(old_code, new_code)
    with open(app_path, "w") as f:
        f.write(code)
    print("App.py updated successfully!")
else:
    print("Old code not found. No changes made.")

App.py updated successfully!


In [70]:
process.terminate()
process.wait()

import subprocess
import time
import os

os.chdir("/content/drive/MyDrive/AI Academic Assistant")

log_file = open("/content/streamlit.log", "w")

process = subprocess.Popen(
    [
        "streamlit", "run", "app.py",
        "--server.port", "8501",
        "--server.address", "0.0.0.0",
        "--server.enableCORS", "false",
        "--server.enableXsrfProtection", "false",
        "--server.headless", "true"
    ],
    stdout=log_file,
    stderr=subprocess.STDOUT
)

time.sleep(10)
print("Streamlit server running:", process.poll() is None)

Streamlit server running: True


In [71]:
from google.colab import output
output.serve_kernel_port_as_iframe(8501)

<IPython.core.display.Javascript object>

In [72]:
path = "/content/drive/MyDrive/AI Academic Assistant/agents.py"

with open(path, "r") as f:
    code = f.read()

start = code.find("class AnswerAgent:")
end = code.find("class EvaluationAgent:")

print(code[start:end])

class AnswerAgent:
    def __init__(self, client):
        self.client = client

    def generate_answer(self, question, context):
        if not context:
            return "This topic is not available in the selected subject syllabus."

        syllabus_context = "\n\n".join(context)

        prompt = f"""
You are an AI Academic Tutor.
Answer the student's question using the selected subject syllabus context.

Rules:
1. First check whether the topic is related to the selected syllabus context.
2. If the topic is unrelated to the selected subject, reply exactly:
   "This topic is not available in the selected subject syllabus."
3. If the syllabus contains only a heading or brief mention, explain that topic
   using standard academic knowledge, while staying within the subject.
4. Give a clear, student-friendly answer with suitable headings and examples.
5. Do not claim that information is directly quoted from the syllabus unless
   it appears in the supplied context.

Selected syllabu

In [73]:
path = "/content/drive/MyDrive/AI Academic Assistant/agents.py"

with open(path, "r") as f:
    code = f.read()

start = code.index("class AnswerAgent:")
end = code.index("class EvaluationAgent:")

new_answer_agent = '''
class AnswerAgent:
    def __init__(self, client):
        self.client = client

    def generate_answer(self, question, context):
        unavailable = "This topic is not available in the selected subject syllabus."

        if not context:
            return unavailable

        syllabus_context = "\\n\\n".join(context)

        prompt = f"""
You are a strict syllabus-based academic assistant.

Use ONLY the information in the syllabus context below.
Never use general knowledge or outside information.
If the answer is not clearly supported by the context, reply exactly:
{unavailable}

If the context contains only a heading without an explanation,
reply exactly:
{unavailable}

Syllabus context:
{syllabus_context}

Student question:
{question}
"""

        response = requests.post(
            f"{NEXUS_BASE_URL}/chat/completions",
            headers={
                "Authorization": f"Bearer {self.client}",
                "Content-Type": "application/json"
            },
            json={
                "model": "nova-micro",
                "messages": [{"role": "user", "content": prompt}],
                "temperature": 0
            },
            timeout=120
        )

        response.raise_for_status()
        result = response.json()

        return result["choices"][0]["message"]["content"] or unavailable


'''

code = code[:start] + new_answer_agent + code[end:]

with open(path, "w") as f:
    f.write(code)

print("AnswerAgent updated successfully!")

AnswerAgent updated successfully!


In [74]:
process.terminate()
process.wait()

import subprocess
import time
import os

os.chdir("/content/drive/MyDrive/AI Academic Assistant")

log_file = open("/content/streamlit.log", "w")

process = subprocess.Popen(
    [
        "streamlit", "run", "app.py",
        "--server.port", "8501",
        "--server.address", "0.0.0.0",
        "--server.enableCORS", "false",
        "--server.enableXsrfProtection", "false",
        "--server.headless", "true"
    ],
    stdout=log_file,
    stderr=subprocess.STDOUT
)

time.sleep(10)
print("Server running:", process.poll() is None)

Server running: True


In [75]:
import chromadb

db_path = "/content/drive/MyDrive/AI Academic Assistant/chroma_db"

db = chromadb.PersistentClient(path=db_path)
collection = db.get_collection("academic_notes")

print("Total syllabus chunks:", collection.count())

results = collection.query(
    query_texts=["Local Search Algorithms"],
    n_results=5
)

for doc in results["documents"][0]:
    print("\n--- SYLLABUS CONTENT ---\n")
    print(doc[:1000])

Total syllabus chunks: 10


/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:02<00:00, 27.9MiB/s]


InvalidArgumentError: Collection expecting embedding with dimension of 1536, got 384

In [76]:
from google.colab import userdata
import requests

api_key = userdata.get("NEXUS_API_KEY")

response = requests.post(
    "https://nexusapi.navigatelabs.ai/v1/embeddings",
    headers={
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json"
    },
    json={
        "model": "text-embedding-3-small",
        "input": "Local Search Algorithms"
    }
)

print("Status:", response.status_code)

if response.status_code == 200:
    embedding = response.json()["data"][0]["embedding"]
    print("Embedding dimension:", len(embedding))
else:
    print(response.text)

Status: 200
Embedding dimension: 1536


In [78]:
import chromadb
import requests
from google.colab import userdata

api_key = userdata.get("NEXUS_API_KEY")

db = chromadb.PersistentClient(
    path="/content/drive/MyDrive/AI Academic Assistant/chroma_db"
)

collection = db.get_collection("academic_notes")

response = requests.post(
    "https://nexusapi.navigatelabs.ai/v1/embeddings",
    headers={
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json"
    },
    json={
        "model": "text-embedding-3-small",
        "input": "Local Search Algorithms"
    }
)

response.raise_for_status()

embedding = response.json()["data"][0]["embedding"]

results = collection.query(
    query_embeddings=[embedding],
    n_results=5,
    include=["documents", "metadatas", "distances"]
)

for doc, metadata, distance in zip(
    results["documents"][0],
    results["metadatas"][0],
    results["distances"][0]
):
    print("\nCONTENT:", doc[:500])
    print("METADATA:", metadata)
    print("DISTANCE:", distance)


CONTENT: or NLP that use Artificial Intelligence 
 
 
 
 
Course 
Outcome 
At the end of the course, the student should be able to, Knowledge Level 
CO1: Understand the principles of Artificial Intelligence.  K2 
CO2: Use appropriate search algorithms for any AI problem.  K2 
CO3: Represent a problem using first order and predicate logic.  K3 
CO4: Provide the apt agent strategy to solve a given problem.  K3 
CO5: Design software agents to solve a problem K3 
 CO/PO Mapping 
(3/2/1indicatesstrengthofcorr
METADATA: {'chunk_index': 1, 'subject_name': 'AI', 'subject': '3Rd Year|Semester 5|AI syllabus', 'semester': 'Semester 5', 'source': 'AI syllabus.pdf', 'year': '3Rd Year'}
DISTANCE: 1.4011025428771973

CONTENT:  Agents – Trust and Reputation in Multi-agent systems. 
Unit– V APPLICATIONS Periods 9 
AI applications – Language Models – Information Retrieval- Information Extraction – Natural Language 
Processing – Machine Translation – Speech Recognition. 
TotalPeriods 45 
Text Books: 
1.

In [79]:
path = "/content/drive/MyDrive/AI Academic Assistant/agents.py"

with open(path, "r") as f:
    code = f.read()

start = code.index("class RetrievalAgent:")
end = code.index("class AnswerAgent:")

print(code[start:end])

class RetrievalAgent:
    def retrieve_context(
        self, question, client, collection, subject_name, year, semester
    ):
        response = requests.post(
            f"{NEXUS_BASE_URL}/embeddings",
            headers={
                "Authorization": f"Bearer {client}",
                "Content-Type": "application/json"
            },
            json={
                "model": "text-embedding-3-small",
                "input": question
            },
            timeout=60
        )

        response.raise_for_status()
        question_embedding = response.json()["data"][0]["embedding"]

        results = collection.query(
            query_embeddings=[question_embedding],
            n_results=5,
            where={
                "$and": [
                    {"subject": subject_name},
                    {"year": year},
                    {"semester": semester}
                ]
            }
        )

        documents = results.get("documents", [[]])[0]
        metad

In [80]:
path = "/content/drive/MyDrive/AI Academic Assistant/agents.py"

with open(path, "r") as f:
    code = f.read()

start = code.index("class RetrievalAgent:")
end = code.index("class AnswerAgent:")

new_retrieval_agent = '''
class RetrievalAgent:
    def retrieve_context(
        self, question, client, collection, subject_name, year, semester
    ):
        response = requests.post(
            f"{NEXUS_BASE_URL}/embeddings",
            headers={
                "Authorization": f"Bearer {client}",
                "Content-Type": "application/json"
            },
            json={
                "model": "text-embedding-3-small",
                "input": question
            },
            timeout=60
        )

        response.raise_for_status()
        question_embedding = response.json()["data"][0]["embedding"]

        results = collection.query(
            query_embeddings=[question_embedding],
            n_results=min(10, collection.count()),
            where={
                "$and": [
                    {"subject": subject_name},
                    {"year": year},
                    {"semester": semester}
                ]
            },
            include=["documents", "metadatas", "distances"]
        )

        documents = results.get("documents", [[]])[0]
        metadatas = results.get("metadatas", [[]])[0]

        sources = sorted({
            metadata["source"]
            for metadata in metadatas
            if metadata and metadata.get("source")
        })

        return documents, sources


'''

code = code[:start] + new_retrieval_agent + code[end:]

with open(path, "w") as f:
    f.write(code)

print("RetrievalAgent updated successfully!")

RetrievalAgent updated successfully!


In [81]:
import ast

path = "/content/drive/MyDrive/AI Academic Assistant/agents.py"

with open(path, "r") as f:
    code = f.read()

ast.parse(code)

print("agents.py syntax is correct!")

agents.py syntax is correct!


In [82]:
process.terminate()
process.wait()

import subprocess
import time
import os

os.chdir("/content/drive/MyDrive/AI Academic Assistant")

log_file = open("/content/streamlit.log", "w")

process = subprocess.Popen(
    [
        "streamlit", "run", "app.py",
        "--server.port", "8501",
        "--server.address", "0.0.0.0",
        "--server.enableCORS", "false",
        "--server.enableXsrfProtection", "false",
        "--server.headless", "true"
    ],
    stdout=log_file,
    stderr=subprocess.STDOUT
)

time.sleep(10)
print("Server running:", process.poll() is None)

Server running: True


In [83]:
path = "/content/drive/MyDrive/AI Academic Assistant/agents.py"

with open(path, "r") as f:
    code = f.read()

old_filter = '{"subject": subject_name}'
new_filter = '{"subject_name": subject_name}'

if old_filter in code:
    code = code.replace(old_filter, new_filter)

    with open(path, "w") as f:
        f.write(code)

    print("Subject filter fixed successfully!")
else:
    print("Old filter not found. No changes made.")

Subject filter fixed successfully!


In [84]:
process.terminate()
process.wait()

import subprocess
import time
import os

os.chdir("/content/drive/MyDrive/AI Academic Assistant")

log_file = open("/content/streamlit.log", "w")

process = subprocess.Popen(
    [
        "streamlit", "run", "app.py",
        "--server.port", "8501",
        "--server.address", "0.0.0.0",
        "--server.enableCORS", "false",
        "--server.enableXsrfProtection", "false",
        "--server.headless", "true"
    ],
    stdout=log_file,
    stderr=subprocess.STDOUT
)

time.sleep(10)
print("Server running:", process.poll() is None)

Server running: True


In [85]:
import chromadb

db = chromadb.PersistentClient(
    path="/content/drive/MyDrive/AI Academic Assistant/chroma_db"
)

collection = db.get_collection("academic_notes")

print("Total chunks:", collection.count())

print("\n--- FILTER TEST ---")

results = collection.get(
    where={
        "$and": [
            {"subject_name": "AI"},
            {"year": "3Rd Year"},
            {"semester": "Semester 5"}
        ]
    },
    include=["metadatas", "documents"]
)

print("Matching chunks:", len(results["ids"]))

for doc, meta in zip(results["documents"], results["metadatas"]):
    print("\nMETADATA:", meta)
    print("CONTENT:", doc[:500])

Total chunks: 10

--- FILTER TEST ---
Matching chunks: 6

METADATA: {'chunk_index': 0, 'subject': '3Rd Year|Semester 5|AI syllabus', 'semester': 'Semester 5', 'subject_name': 'AI', 'year': '3Rd Year', 'source': 'AI syllabus.pdf'}
CONTENT:  
 
 
4  
VIVEKANANDHA COLLEGE OF ENGINEERING FOR WOMEN 
(Autonomous Institution, Affiliated to Anna University 
,Chennai)Elayampalayam, Tiruchengode – 637 205  
Programme B.E. /B.TECH ProgrammeCode  Regulation 2023 
Department CSE & IT Semester V 
 
CourseCode 
 
Course Name 
PeriodsPer 
Week Credit Maximum Marks 
L T P C CA ESE Total 
U23IT512 Artificial Intelligence 3 0 0 3 40 60 100 
 
 
 
Course 
Objective 
The main objective of this course is to: 
• Understand the various characteristics o

METADATA: {'chunk_index': 1, 'subject': '3Rd Year|Semester 5|AI syllabus', 'subject_name': 'AI', 'semester': 'Semester 5', 'source': 'AI syllabus.pdf', 'year': '3Rd Year'}
CONTENT: or NLP that use Artificial Intelligence 
 
 
 
 
Course 
Outcome 
At the end o

In [86]:
import chromadb

db = chromadb.PersistentClient(
    path="/content/drive/MyDrive/AI Academic Assistant/chroma_db"
)

collection = db.get_collection("academic_notes")

results = collection.get(
    where={"subject_name": "AI"},
    include=["documents", "metadatas"]
)

for doc, meta in zip(results["documents"], results["metadatas"]):
    if "Local Search" in doc or "local search" in doc.lower():
        print("FOUND IN CHUNK:", meta["chunk_index"])
        print("\nCONTENT:\n", doc)

print("\nSearch completed.")

FOUND IN CHUNK: 2

CONTENT:
  2 1 -         3 1 
CO3 3 2 1 1         3 1 
CO4 3 2 1 1         3 1 
CO5 3 2 1 1         3 1 
Course Assessment Methods 
 Direct  
1. Continuous Assessment TestI,II&III 
2. Assignment 
3. End-Semester examinations 
Indirect 
1.Course-end survey 

 
Content of the syllabus 
Unit– I INTRODUCTION TO ARTIFICIAL INTELLIGENCE Periods 9 
Introduction–Definition – Future of Artificial Intelligence – Characteristics of Intelligent Agents–
Typical Intelligent Agents – Problem Solving Approach to Typical AI problems. 
Unit-II PROBLEM SOLVING METHODS Periods 9 
Problem solving Methods – Search Strategies- Uninformed – Informed – Heuristics – Local Search 
Algorithms and Optimization Problems – Searching with Partial Observations – Constraint Satisfaction 
Problems – Constraint Propagation – Backtrack

Search completed.


In [87]:
from google.colab import userdata
import requests

api_key = userdata.get("NEXUS_API_KEY")

response = requests.post(
    "https://nexusapi.navigatelabs.ai/v1/embeddings",
    headers={
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json"
    },
    json={
        "model": "text-embedding-3-small",
        "input": "Local Search Algorithms"
    }
)

embedding = response.json()["data"][0]["embedding"]

results = collection.query(
    query_embeddings=[embedding],
    n_results=10,
    where={
        "$and": [
            {"subject_name": "AI"},
            {"year": "3Rd Year"},
            {"semester": "Semester 5"}
        ]
    },
    include=["documents", "metadatas", "distances"]
)

for doc, meta, distance in zip(
    results["documents"][0],
    results["metadatas"][0],
    results["distances"][0]
):
    print("\nCHUNK:", meta["chunk_index"])
    print("DISTANCE:", distance)
    print("CONTAINS LOCAL SEARCH:", "local search" in doc.lower())

TimeoutException: Requesting secret NEXUS_API_KEY timed out. Secrets can only be fetched when running from the Colab UI.

In [88]:
import os

api_key = os.environ.get("NEXUS_API_KEY")

print("API key available:", bool(api_key))

API key available: True


In [89]:
import os
import requests

api_key = os.environ["NEXUS_API_KEY"]

response = requests.post(
    "https://nexusapi.navigatelabs.ai/v1/embeddings",
    headers={
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json"
    },
    json={
        "model": "text-embedding-3-small",
        "input": "Local Search Algorithms"
    },
    timeout=60
)

response.raise_for_status()
embedding = response.json()["data"][0]["embedding"]

results = collection.query(
    query_embeddings=[embedding],
    n_results=10,
    where={
        "$and": [
            {"subject_name": "AI"},
            {"year": "3Rd Year"},
            {"semester": "Semester 5"}
        ]
    },
    include=["documents", "metadatas", "distances"]
)

for doc, meta, distance in zip(
    results["documents"][0],
    results["metadatas"][0],
    results["distances"][0]
):
    print("\nCHUNK:", meta["chunk_index"])
    print("DISTANCE:", distance)
    print("CONTAINS LOCAL SEARCH:", "local search" in doc.lower())


CHUNK: 1
DISTANCE: 1.4012433290481567
CONTAINS LOCAL SEARCH: False

CHUNK: 4
DISTANCE: 1.5251020193099976
CONTAINS LOCAL SEARCH: False

CHUNK: 5
DISTANCE: 1.5255531072616577
CONTAINS LOCAL SEARCH: False

CHUNK: 2
DISTANCE: 1.578568935394287
CONTAINS LOCAL SEARCH: True

CHUNK: 3
DISTANCE: 1.5909364223480225
CONTAINS LOCAL SEARCH: False

CHUNK: 0
DISTANCE: 1.6954995393753052
CONTAINS LOCAL SEARCH: False


In [90]:
path = "/content/drive/MyDrive/AI Academic Assistant/agents.py"

with open(path, "r") as f:
    code = f.read()

start = code.index("class AnswerAgent:")
end = code.index("class EvaluationAgent:")

new_answer_agent = '''
class AnswerAgent:
    def __init__(self, client):
        self.client = client

    def generate_answer(self, question, context):

        unavailable = "This topic is not available in the selected subject syllabus."

        if not context:
            return unavailable

        syllabus_context = "\\\\n\\\\n".join(context)

        prompt = f"""
You are an AI Academic Tutor for students.

Your task is to answer questions based on the selected subject syllabus.

STRICT RULES:

1. First check whether the requested topic is mentioned in the syllabus context.

2. If the topic is explicitly mentioned in the syllabus context,
   provide a clear and detailed academic explanation of that topic.

3. When the syllabus mentions only a topic heading or keyword,
   you MAY use standard academic knowledge to explain that exact topic.

4. Do not introduce unrelated topics or concepts that are outside
   the selected syllabus.

5. If the requested topic is not mentioned or supported by the
   syllabus context, reply exactly:
   "{unavailable}"

6. If the question is unrelated to the selected subject,
   reply exactly:
   "{unavailable}"

7. Explain in simple, student-friendly language with suitable
   headings and examples.

8. Do not claim that the complete explanation is directly taken
   from the syllabus.

SELECTED SYLLABUS CONTEXT:
{syllabus_context}

STUDENT QUESTION:
{question}
"""

        response = requests.post(
            f"{NEXUS_BASE_URL}/chat/completions",
            headers={
                "Authorization": f"Bearer {self.client}",
                "Content-Type": "application/json"
            },
            json={
                "model": "nova-micro",
                "messages": [
                    {"role": "user", "content": prompt}
                ],
                "temperature": 0
            },
            timeout=120
        )

        response.raise_for_status()
        result = response.json()

        return (
            result["choices"][0]["message"]["content"]
            or unavailable
        )


'''

code = code[:start] + new_answer_agent + code[end:]

with open(path, "w") as f:
    f.write(code)

print("AnswerAgent corrected successfully!")

AnswerAgent corrected successfully!


In [91]:
process.terminate()
process.wait()
print("Old Streamlit server stopped!")

Old Streamlit server stopped!


In [92]:
import subprocess
import time
import os

os.chdir("/content/drive/MyDrive/AI Academic Assistant")

log_file = open("/content/streamlit.log", "w")

process = subprocess.Popen(
    [
        "streamlit", "run", "app.py",
        "--server.port", "8501",
        "--server.address", "0.0.0.0",
        "--server.enableCORS", "false",
        "--server.enableXsrfProtection", "false",
        "--server.headless", "true"
    ],
    stdout=log_file,
    stderr=subprocess.STDOUT
)

time.sleep(10)
print("Server running:", process.poll() is None)

Server running: True


In [93]:
from google.colab import output
output.serve_kernel_port_as_iframe(8501)

<IPython.core.display.Javascript object>

In [94]:
path = "/content/drive/MyDrive/AI Academic Assistant/app.py"

with open(path, "r") as f:
    lines = f.readlines()

for i, line in enumerate(lines):
    if any(word in line.lower() for word in [
        "retrieve_context",
        "generate_answer",
        "context =",
        "answer ="
    ]):
        start = max(0, i - 5)
        end = min(len(lines), i + 12)
        print(f"\n--- Lines {start+1} to {end} ---")
        print("".join(lines[start:end]))


--- Lines 289 to 305 ---
        },
        include=["metadatas"]
    )
    return len(result["ids"])

def generate_answer(question):
    try:
        year, semester, subject_key = current_scope()

        processed_question = query_agent.process_query(question)

        if not processed_question:
            return "Please enter a valid question.", []

        if get_subject_count() == 0:
            return "Please process this subject's syllabus first.", []



--- Lines 301 to 317 ---
            return "Please enter a valid question.", []

        if get_subject_count() == 0:
            return "Please process this subject's syllabus first.", []

        context, sources = retrieval_agent.retrieve_context(
            processed_question,
            client,
            collection,
            subject_key,
            year,
            semester
        )

        answer = answer_agent.generate_answer(
            processed_question, context
        )


--- Lines 310 to 326 ---
     

In [95]:
path = "/content/drive/MyDrive/AI Academic Assistant/agents.py"

with open(path, "r") as f:
    code = f.read()

for class_name in ["class AnswerAgent:", "class EvaluationAgent:"]:
    start = code.index(class_name)
    next_class = code.find("\nclass ", start + 1)
    if next_class == -1:
        next_class = len(code)

    print("\n" + "=" * 50)
    print(code[start:next_class])


class AnswerAgent:
    def __init__(self, client):
        self.client = client

    def generate_answer(self, question, context):

        unavailable = "This topic is not available in the selected subject syllabus."

        if not context:
            return unavailable

        syllabus_context = "\\n\\n".join(context)

        prompt = f"""
You are an AI Academic Tutor for students.

Your task is to answer questions based on the selected subject syllabus.

STRICT RULES:

1. First check whether the requested topic is mentioned in the syllabus context.

2. If the topic is explicitly mentioned in the syllabus context,
   provide a clear and detailed academic explanation of that topic.

3. When the syllabus mentions only a topic heading or keyword,
   you MAY use standard academic knowledge to explain that exact topic.

4. Do not introduce unrelated topics or concepts that are outside
   the selected syllabus.

5. If the requested topic is not mentioned or supported by the
   syllabu

In [96]:
import chromadb

db = chromadb.PersistentClient(
    path="/content/drive/MyDrive/AI Academic Assistant/chroma_db"
)
collection = db.get_collection("academic_notes")

result = collection.get(
    where={"subject_name": "AI"},
    include=["documents", "metadatas"]
)

test_context = [
    doc for doc, meta in zip(result["documents"], result["metadatas"])
    if "local search" in doc.lower()
]

print("Matching chunks:", len(test_context))
print("Context contains Local Search:", bool(test_context))

if test_context:
    test_answer = answer_agent.generate_answer(
        "Explain Local Search Algorithms",
        test_context
    )
    print("\nANSWER:\n", test_answer)

Matching chunks: 1
Context contains Local Search: True


NameError: name 'answer_agent' is not defined

In [97]:
import os
import sys

project_path = "/content/drive/MyDrive/AI Academic Assistant"
if project_path not in sys.path:
    sys.path.insert(0, project_path)

from agents import AnswerAgent

api_key = os.environ.get("NEXUS_API_KEY")

if not api_key:
    print("Nexus API key is not available in this session.")
else:
    answer_agent = AnswerAgent(api_key)

    test_answer = answer_agent.generate_answer(
        "Explain Local Search Algorithms",
        test_context
    )

    print("ANSWER:\n", test_answer)

ANSWER:
 Sure, let's dive into **Local Search Algorithms**! This topic is part of the syllabus under **Unit-II: Problem Solving Methods**.

### **Local Search Algorithms**

#### **What is Local Search?**
Local search algorithms are a class of optimization methods used to find the best solution in a large, often complex, solution space. They start with an initial solution and iteratively improve it by making small changes (or "moves") to the solution.

#### **How Do They Work?**
1. **Initial Solution**: Begin with an initial solution, which can be random or chosen based on some heuristic.
2. **Neighborhood**: Define a neighborhood of solutions around the current solution. This means identifying what small changes can be made to the current solution.
3. **Evaluation**: Evaluate the current solution and its neighbors using a given objective function (which measures how good a solution is).
4. **Move**: Choose the best neighbor (the one that gives the best value according to the objective 

In [98]:
path = "/content/drive/MyDrive/AI Academic Assistant/app.py"

with open(path, "r") as f:
    lines = f.readlines()

for i, line in enumerate(lines):
    if "answer_agent" in line or "retrieval_agent" in line:
        start = max(0, i - 5)
        end = min(len(lines), i + 6)
        print(f"\n--- Lines {start+1} to {end} ---")
        print("".join(lines[start:end]))


--- Lines 42 to 52 ---
# --------------------------------------------------
chroma_client = chromadb.PersistentClient(path=str(BASE_DIR / "chroma_db"))
collection = chroma_client.get_or_create_collection(name="academic_notes")

query_agent = QueryAgent()
retrieval_agent = RetrievalAgent()
answer_agent = AnswerAgent(client)
evaluation_agent = EvaluationAgent()

# --------------------------------------------------
# 4. DISCOVER YEARS, SEMESTERS AND SUBJECTS


--- Lines 43 to 53 ---
chroma_client = chromadb.PersistentClient(path=str(BASE_DIR / "chroma_db"))
collection = chroma_client.get_or_create_collection(name="academic_notes")

query_agent = QueryAgent()
retrieval_agent = RetrievalAgent()
answer_agent = AnswerAgent(client)
evaluation_agent = EvaluationAgent()

# --------------------------------------------------
# 4. DISCOVER YEARS, SEMESTERS AND SUBJECTS
# Supports both:


--- Lines 301 to 311 ---
            return "Please enter a valid question.", []

        if get_subject_count(

In [99]:
path = "/content/drive/MyDrive/AI Academic Assistant/app.py"

with open(path, "r") as f:
    lines = f.readlines()

for start, end in [(90, 180), (270, 305)]:
    print(f"\n--- app.py lines {start}-{end} ---")
    print("".join(lines[start-1:end]))


--- app.py lines 90-180 ---
                            "name": display_subject(subject_dir.name),
                            "folder_name": subject_dir.name,
                            "path": subject_dir
                        })

            # Direct PDF/DOCX files (e.g. DBMS syllabus.pdf)
            direct_files = list(semester_dir.glob("*.pdf")) + list(semester_dir.glob("*.docx"))
            for file_path in sorted(direct_files):
                subject_name = file_path.stem
                # Remove common suffixes such as "syllabus"
                subject_name = subject_name.replace("_", " ").replace("-", " ").strip()
                import re
                subject_name = re.sub(r"\s*syllabus\s*$", "", subject_name, flags=re.IGNORECASE).strip()
                if not subject_name:
                    subject_name = file_path.stem
                subjects.append({
                    "name": subject_name.upper() if len(subject_name) <= 5 else subject_name.title(),
       

In [100]:
path = "/content/drive/MyDrive/AI Academic Assistant/app.py"

with open(path, "r") as f:
    lines = f.readlines()

for i, line in enumerate(lines):
    if "def current_scope" in line:
        print("".join(lines[i:i+25]))
        break

def current_scope():
    return (
        st.session_state.selected_year,
        st.session_state.selected_semester,
        st.session_state.selected_subject_key
    )

def get_subject_files(subject_info):
    if "files" in subject_info:
        return subject_info["files"]
    folder = subject_info["path"]
    return sorted(list(folder.glob("*.pdf")) + list(folder.glob("*.docx")))

# --------------------------------------------------
# 7. PROCESS SELECTED SUBJECT
# --------------------------------------------------
def process_subject(subject_info):
    year, semester, subject_key = current_scope()
    files = get_subject_files(subject_info)
    if not files:
        return 0, "No PDF or DOCX files found."

    total_chunks = 0
    empty_files = []




In [101]:
path = "/content/drive/MyDrive/AI Academic Assistant/agents.py"

with open(path, "r") as f:
    code = f.read()

old = '{"subject_name": subject_name},'
new = '{"subject": subject_name},'

if old in code:
    code = code.replace(old, new, 1)

    with open(path, "w") as f:
        f.write(code)

    print("Changed successfully!")
else:
    print("Expected line not found. No changes made.")

Changed successfully!


In [102]:
process.terminate()
process.wait()
print("Old server stopped!")

Old server stopped!


In [103]:
import subprocess
import time
import os

os.chdir("/content/drive/MyDrive/AI Academic Assistant")

log_file = open("/content/streamlit.log", "w")

process = subprocess.Popen(
    [
        "streamlit", "run", "app.py",
        "--server.port", "8501",
        "--server.address", "0.0.0.0",
        "--server.enableCORS", "false",
        "--server.enableXsrfProtection", "false",
        "--server.headless", "true"
    ],
    stdout=log_file,
    stderr=subprocess.STDOUT
)

time.sleep(10)
print("Server running:", process.poll() is None)

Server running: True


In [15]:
from google.colab import output
output.serve_kernel_port_as_iframe(8501)

<IPython.core.display.Javascript object>

In [105]:
path = "/content/drive/MyDrive/AI Academic Assistant/agents.py"

with open(path, "r") as f:
    code = f.read()

old = '''5. If the requested topic is not mentioned or supported by the
   syllabus context, reply exactly:
   "{unavailable}"

6. If the question is unrelated to the selected subject,
   reply exactly:
   "{unavailable}"'''

new = '''5. If the requested topic is not mentioned or supported by the
   syllabus context, reply with ONLY this exact sentence:
   "{unavailable}"

6. If the question is unrelated to the selected subject,
   reply with ONLY this exact sentence:
   "{unavailable}"

7. Never provide additional explanations, suggestions,
   examples, or comments when returning the unavailable message.'''

if old in code:
    code = code.replace(old, new, 1)

    with open(path, "w") as f:
        f.write(code)

    print("Unavailable response rule updated successfully!")
else:
    print("Expected prompt text not found. No changes made.")

Unavailable response rule updated successfully!


In [106]:
process.terminate()
process.wait()
print("Old server stopped!")

Old server stopped!


In [107]:
import subprocess
import time
import os

os.chdir("/content/drive/MyDrive/AI Academic Assistant")

log_file = open("/content/streamlit.log", "w")

process = subprocess.Popen(
    [
        "streamlit", "run", "app.py",
        "--server.port", "8501",
        "--server.address", "0.0.0.0",
        "--server.enableCORS", "false",
        "--server.enableXsrfProtection", "false",
        "--server.headless", "true"
    ],
    stdout=log_file,
    stderr=subprocess.STDOUT
)

time.sleep(10)
print("Server running:", process.poll() is None)

Server running: True


In [3]:
from google.colab import output
output.serve_kernel_port_as_iframe(8501)

<IPython.core.display.Javascript object>

In [1]:
from google.colab import output
output.serve_kernel_port_as_iframe(8501)

<IPython.core.display.Javascript object>

In [4]:
py -m streamlit run app.py

SyntaxError: invalid syntax (2774303893.py, line 1)

In [5]:
!pip install streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 12.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 14.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [streamlit]


In [7]:
!streamlit run app.py &>/content/streamlit.log &

In [8]:
import os

print(os.listdir('/content'))

['.config', 'streamlit.log', 'sample_data']


In [9]:
import os
import zipfile
import glob

zip_files = glob.glob("/content/*.zip")

print("ZIP files found:", zip_files)

if zip_files:
    zip_path = zip_files[0]

    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall("/content/project")

    print("Project extracted successfully!")
    print(os.listdir("/content/project"))
else:
    print("ZIP file not found!")

ZIP files found: ['/content/AI-Powered Acadameic Knowledge.zip.zip']


BadZipFile: File is not a zip file

In [10]:
import os

path = "/content/AI-Powered Acadameic Knowledge.zip.zip"

print("File size:", os.path.getsize(path), "bytes")

with open(path, "rb") as f:
    print("File signature:", f.read(10))

File size: 11534336 bytes
File signature: b'PK\x03\x04\x14\x00\x00\x00\x00\x00'


In [11]:
import zipfile
import os

path = "/content/AI-Powered Acadameic Knowledge.zip.zip"

print("Is valid ZIP:", zipfile.is_zipfile(path))

try:
    with zipfile.ZipFile(path, 'r') as z:
        print("Files inside ZIP:")
        print(z.namelist())
except Exception as e:
    print("Error:", e)

Is valid ZIP: False
Error: File is not a zip file


In [12]:
import zipfile

path = "/content/AI-Powered Academic Knowledge.zip"

print(zipfile.is_zipfile(path))

False


In [14]:
import os

path = "/content/AI-Powered Acadameic Knowledge.zip.zip"

with open(path, "rb") as f:
    data = f.read(100)

print("First 100 bytes:", data)
print("File size:", os.path.getsize(path))
print("Last 30 bytes:")

with open(path, "rb") as f:
    f.seek(-30, 2)
    print(f.read())

FileNotFoundError: [Errno 2] No such file or directory: '/content/AI-Powered Acadameic Knowledge.zip.zip'

In [15]:
import os

print(os.listdir('/content'))

['.config', 'streamlit.log', 'AI-Powered Acadameic Knowledge.zip', '.ipynb_checkpoints', 'sample_data']


In [16]:
import os
import zipfile

path = "/content/AI-Powered Acadameic Knowledge.zip"

print("File exists:", os.path.exists(path))
print("Is valid ZIP:", zipfile.is_zipfile(path))

File exists: True
Is valid ZIP: False


In [17]:
import os

path = "/content/AI-Powered Acadameic Knowledge.zip"

print("File Size:", os.path.getsize(path), "bytes")

with open(path, "rb") as f:
    print("File Signature:", f.read(10))

File Size: 8388608 bytes
File Signature: b'PK\x03\x04\x14\x00\x00\x00\x00\x00'


In [18]:
import zipfile

path = "/content/AI-Powered Acadameic Knowledge.zip"

print(zipfile.is_zipfile(path))

False


In [19]:
import zipfile

path = "/content/AI-Powered Acadameic Knowledge.zip"

print(zipfile.is_zipfile(path))

True


In [20]:
path = "/content/AI-Powered Acadameic Knowledge.zip"

with open(path, "rb") as f:
    print(repr(f.read(20)))

b'PK\x03\x04\x14\x00\x00\x00\x00\x00\xb8Et]\x00\x00\x00\x00\x00\x00'


In [21]:
import os
print(os.listdir('/content'))

['.config', 'streamlit.log', 'AI-Powered Acadameic Knowledge.zip', '.ipynb_checkpoints', 'sample_data']


In [23]:
import os

for root, dirs, files in os.walk('/content'):
    for file in files:
        if file == 'app.py':
            print(os.path.join(root, file))

In [24]:
!pip install streamlit

In [26]:
!streamlit run /content/app.py &>/content/streamlit.log &

In [27]:
import os

print("Files in Colab:")
print(os.listdir('/content'))

print("\nSearching for app.py:")

for root, dirs, files in os.walk('/content'):
    if 'app.py' in files:
        print("Found:", os.path.join(root, 'app.py'))

Files in Colab:
['.config', 'streamlit.log', 'AI-Powered Acadameic Knowledge.zip', '.ipynb_checkpoints', 'sample_data']

Searching for app.py:


In [28]:
from google.colab import files

uploaded = files.upload()

Saving AI-Powered Acadameic Knowledge.zip to AI-Powered Acadameic Knowledge (1).zip


In [29]:
import zipfile
import os

path = "/content/AI-Powered Acadameic Knowledge (1).zip"

print("File exists:", os.path.exists(path))
print("Valid ZIP:", zipfile.is_zipfile(path))

File exists: True
Valid ZIP: True


In [30]:
import zipfile
import os

zip_path = "/content/AI-Powered Acadameic Knowledge (1).zip"
extract_path = "/content/project"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Extraction completed!")
print(os.listdir(extract_path))

Extraction completed!
['AI-Powered Acadameic Knowledge']


In [31]:
import os

for root, dirs, files in os.walk('/content/project'):
    if 'app.py' in files:
        print("APP FILE:", os.path.join(root, 'app.py'))

APP FILE: /content/project/AI-Powered Acadameic Knowledge/app.py


In [32]:
import os

project_path = "/content/project/AI-Powered Acadameic Knowledge"

for root, dirs, files in os.walk(project_path):
    for file in files:
        print(os.path.join(root, file))

/content/project/AI-Powered Acadameic Knowledge/.env
/content/project/AI-Powered Acadameic Knowledge/test_embedding.py
/content/project/AI-Powered Acadameic Knowledge/app.py
/content/project/AI-Powered Acadameic Knowledge/test_retrieval.py
/content/project/AI-Powered Acadameic Knowledge/agents - Copy.py
/content/project/AI-Powered Acadameic Knowledge/test_generation.py
/content/project/AI-Powered Acadameic Knowledge/agents.py
/content/project/AI-Powered Acadameic Knowledge/.gitignore
/content/project/AI-Powered Acadameic Knowledge/test_chunking.py
/content/project/AI-Powered Acadameic Knowledge/requirements.txt
/content/project/AI-Powered Acadameic Knowledge/app_old.py
/content/project/AI-Powered Acadameic Knowledge/test_models.py
/content/project/AI-Powered Acadameic Knowledge/test_rag.py
/content/project/AI-Powered Acadameic Knowledge/app_backup.py
/content/project/AI-Powered Acadameic Knowledge/test_vector_store.py
/content/project/AI-Powered Acadameic Knowledge/notes/high pressure-

In [33]:
%cd "/content/project/AI-Powered Acadameic Knowledge"

/content/project/AI-Powered Acadameic Knowledge


In [34]:
!pip install -r requirements.txt

In [35]:
!streamlit run app.py --server.port 8501 --server.address 0.0.0.0



2026-11-20 04:37:52.418 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://136.64.181.163:8501

  Stopping...
  Stopping...


In [36]:
!npm install -g localtunnel

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏
added 22 packages in 2s
⠏
⠏3 packages are looking for funding
⠏  run `npm fund` for details
⠏

In [37]:
!lt --port 8501

^C


In [38]:
!curl -I http://127.0.0.1:8501

curl: (7) Failed to connect to 127.0.0.1 port 8501 after 0 ms: Couldn't connect to server


In [39]:
%cd "/content/project/AI-Powered Acadameic Knowledge"

!streamlit run app.py --server.port 8501 --server.address 0.0.0.0

/content/project/AI-Powered Acadameic Knowledge


2026-11-20 04:41:44.853 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://136.64.181.163:8501

  Stopping...


In [40]:
!curl -I http://127.0.0.1:8501

curl: (7) Failed to connect to 127.0.0.1 port 8501 after 0 ms: Couldn't connect to server


In [41]:
!lt --port 8501

^C


In [42]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared
!./cloudflared tunnel --url http://localhost:8501

2026-11-20T04:44:36Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-11-20T04:44:36Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-11-20T04:44:40Z INF +--------------------------------------------------------------------------------------------+
2026-11-20T04:44:40Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-11-20T04:44:40Z INF |  https://ridge-inner-them-scenic.trycloudflare.com    

In [43]:
!curl -I http://localhost:8501

curl: (7) Failed to connect to localhost port 8501 after 0 ms: Couldn't connect to server


In [44]:
import subprocess
import time
import requests
import os

os.chdir("/content/project/AI-Powered Acadameic Knowledge")

log = open("/content/streamlit.log", "w")

process = subprocess.Popen(
    ["streamlit", "run", "app.py",
     "--server.port", "8501",
     "--server.address", "0.0.0.0",
     "--server.headless", "true"],
    stdout=log,
    stderr=subprocess.STDOUT
)

time.sleep(10)

try:
    response = requests.get("http://127.0.0.1:8501")
    print("SUCCESS! Streamlit is running.")
    print("Status Code:", response.status_code)
except Exception as e:
    print("ERROR:", e)
    print(open("/content/streamlit.log").read()[-3000:])

SUCCESS! Streamlit is running.
Status Code: 200


In [45]:
!./cloudflared tunnel --url http://localhost:8501

2026-11-20T04:46:54Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-11-20T04:46:54Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-11-20T04:46:57Z INF +--------------------------------------------------------------------------------------------+
2026-11-20T04:46:57Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-11-20T04:46:57Z INF |  https://only-breath-sir-sectors.trycloudflare.com    

In [46]:
import requests

try:
    r = requests.get("http://127.0.0.1:8501")
    print("Streamlit Status:", r.status_code)
except Exception as e:
    print("Streamlit Error:", e)

Streamlit Error: HTTPConnectionPool(host='127.0.0.1', port=8501): Max retries exceeded with url: / (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x7a5c105da990>: Failed to establish a new connection: [Errno 111] Connection refused'))


In [47]:
import subprocess
import time
import requests
import os

os.chdir("/content/project/AI-Powered Acadameic Knowledge")

log = open("/content/streamlit.log", "w")

process = subprocess.Popen(
    ["streamlit", "run", "app.py",
     "--server.port", "8501",
     "--server.address", "0.0.0.0",
     "--server.headless", "true"],
    stdout=log,
    stderr=subprocess.STDOUT
)

time.sleep(10)

try:
    r = requests.get("http://127.0.0.1:8501")
    print("Status:", r.status_code)
except Exception as e:
    print("Error:", e)
    print(open("/content/streamlit.log").read()[-3000:])

Status: 200


In [48]:
!./cloudflared tunnel --protocol http2 --url http://127.0.0.1:8501

2026-11-20T04:49:55Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-11-20T04:49:55Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-11-20T04:49:58Z INF +--------------------------------------------------------------------------------------------+
2026-11-20T04:49:58Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-11-20T04:49:58Z INF |  https://returns-ronald-devices-reducing.trycloudflare

In [49]:
!tail -50 /content/streamlit.log



2026-11-20 04:49:15.599 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://136.64.181.163:8501

────────────────────────── Traceback (most recent call last) ───────────────────────────
  /usr/local/lib/python3.13/dist-packages/streamlit/runtime/scriptrunner/exec_code.py:  
  136 in exec_func_with_error_handling                                                  
                                                                                        
  /usr/local/lib/python3.13/dist-packages/streamlit/runtime/scriptrunner/script_runner  
  .py:936 in code_to_exec                                                               
                                                                                        
  /content/project/AI-Powered Acadameic Knowledge/app.py:6 in <module>                  
                                                

In [50]:
!pip install chromadb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 85.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 84.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 55.9 MB/s  0:00:00
  Attempting uninstall: urllib3
    Found existing installation: urllib3 2.5.0
    Uninstalling urllib3-2.5.0:
      Successfully uninstalled urllib3-2.5.0
  Attempting uninstall: opentelemetry-api
    Found existing installation: opentelemetry-api 1.42.1
    Uninstalling opentelemetry-api-1.42.1:
      Successfully uninstalled opentelemetry-api-1.42.1
  Attempting uninstall: oauthlib
    Found existing installation: oauthlib 3.3.1
    Uninstalling oauthlib-3.3.1:
      Successfully uninstalled oauthlib-3.3.1
  Attempting uninstall: opentelemetry-semantic-conventions
    Found existing installation: opentelemetry-semantic-conventions 0.63b1
    Uninstalling opentelemetry-semantic-conventions-0.63b1:
      Successfully uninstalled opentelemetry-semantic-conventions

In [1]:
!pip install -r "/content/project/AI-Powered Acadameic Knowledge/requirements.txt"

In [2]:
import chromadb
print("ChromaDB installed successfully!")

ChromaDB installed successfully!


In [3]:
%cd "/content/project/AI-Powered Acadameic Knowledge"
!streamlit run app.py --server.port 8501 --server.address 0.0.0.0

/content/project/AI-Powered Acadameic Knowledge


2026-11-20 04:53:16.120 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://136.64.181.163:8501

  Stopping...
  Stopping...


In [4]:
import requests
r = requests.get("http://127.0.0.1:8501")
print("Status:", r.status_code)

ConnectionError: HTTPConnectionPool(host='127.0.0.1', port=8501): Max retries exceeded with url: / (Caused by NewConnectionError("HTTPConnection(host='127.0.0.1', port=8501): Failed to establish a new connection: [Errno 111] Connection refused"))

In [5]:
import os
os.chdir("/content/project/AI-Powered Acadameic Knowledge")

!streamlit run app.py --server.port 8501 --server.address 0.0.0.0



2026-11-20 04:54:28.677 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://136.64.181.163:8501

  Stopping...


In [6]:
import requests
r = requests.get("http://127.0.0.1:8501")
print(r.status_code)

ConnectionError: HTTPConnectionPool(host='127.0.0.1', port=8501): Max retries exceeded with url: / (Caused by NewConnectionError("HTTPConnection(host='127.0.0.1', port=8501): Failed to establish a new connection: [Errno 111] Connection refused"))

In [7]:
import os

for f in ["/content/streamlit.log", "/content/streamlit_debug.log"]:
    if os.path.exists(f):
        print("\n---", f, "---")
        with open(f, "r") as file:
            print(file.read()[-7000:])


--- /content/streamlit.log ---


2026-11-20 04:49:15.599 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://136.64.181.163:8501

────────────────────────── Traceback (most recent call last) ───────────────────────────
  /usr/local/lib/python3.13/dist-packages/streamlit/runtime/scriptrunner/exec_code.py:  
  136 in exec_func_with_error_handling                                                  
                                                                                        
  /usr/local/lib/python3.13/dist-packages/streamlit/runtime/scriptrunner/script_runner  
  .py:936 in code_to_exec                                                               
                                                                                        
  /content/project/AI-Powered Acadameic Knowledge/app.py:6 in <module>                  
                

In [8]:
%pip install chromadb


In [9]:
import chromadb
print("ChromaDB installed successfully!")

ChromaDB installed successfully!


In [10]:
import os
os.chdir("/content/project/AI-Powered Acadamec Knowledge")

FileNotFoundError: [Errno 2] No such file or directory: '/content/project/AI-Powered Acadamec Knowledge'

In [11]:
import os
os.chdir("/content/project/AI-Powered Acadamec Knowledge")

FileNotFoundError: [Errno 2] No such file or directory: '/content/project/AI-Powered Acadamec Knowledge'

In [12]:
import os

print(os.listdir("/content/project"))

['AI-Powered Acadameic Knowledge']


In [14]:
import os
os.chdir("/content/project/AI-Powered Acadameic Knowledge")

!streamlit run app.py --server.port 8501 --server.address 0.0.0.0



2026-11-20 04:59:54.093 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://136.64.181.163:8501

  Stopping...


In [16]:
from google.colab import output
output.serve_kernel_port_as_iframe(8501)

<IPython.core.display.Javascript object>

In [17]:
!curl -I --max-time 5 http://127.0.0.1:8501

curl: (7) Failed to connect to 127.0.0.1 port 8501 after 0 ms: Couldn't connect to server


In [18]:
import subprocess, os

os.chdir("/content/project/AI-Powered Acadameic Knowledge")

log = open("/content/streamlit.log", "w")
process = subprocess.Popen(
    ["streamlit", "run", "app.py", "--server.port", "8501",
     "--server.address", "0.0.0.0"],
    stdout=log,
    stderr=subprocess.STDOUT
)

print("Streamlit starting...")

Streamlit starting...


In [19]:
from google.colab import output
output.serve_kernel_port_as_iframe(8501)

<IPython.core.display.Javascript object>

In [20]:
import requests

try:
    r = requests.get("http://127.0.0.1:8501", timeout=5)
    print("Server status:", r.status_code)
except Exception as e:
    print("Server error:", e)

print("\n--- APP ERROR ---")
print(open("/content/streamlit.log").read()[-4000:])

Server status: 200

--- APP ERROR ---
 Host header: origin=https://8501-m-s-usc1c8-2cjpl8p37rp7s-c.us-central1.prod.r.colab.dev, host=m-s-usc1c8-2cjpl8p37rp7s.us-central1-c.c.codatalab-user-runtimes.internal:8007
2026-11-20 05:05:51.087 Rejecting WebSocket connection with disallowed Origin or Host header: origin=https://8501-m-s-usc1c8-2cjpl8p37rp7s-c.us-central1.prod.r.colab.dev, host=m-s-usc1c8-2cjpl8p37rp7s.us-central1-c.c.codatalab-user-runtimes.internal:8007
2026-11-20 05:06:01.080 Rejecting WebSocket connection with disallowed Origin or Host header: origin=https://8501-m-s-usc1c8-2cjpl8p37rp7s-c.us-central1.prod.r.colab.dev, host=m-s-usc1c8-2cjpl8p37rp7s.us-central1-c.c.codatalab-user-runtimes.internal:8007
2026-11-20 05:06:06.496 Rejecting WebSocket connection with disallowed Origin or Host header: origin=https://8501-m-s-usc1c8-2cjpl8p37rp7s-c.us-central1.prod.r.colab.dev, host=m-s-usc1c8-2cjpl8p37rp7s.us-central1-c.c.codatalab-user-runtimes.internal:8007
2026-11-20 05:06:08.11

In [21]:
import os
os.chdir("/content/project/AI-Powered Acadameic Knowledge")

!streamlit run app.py --server.port 8501 --server.address 0.0.0.0 --server.enableCORS false --server.enableXsrfProtection false



2026-11-20 05:07:21.308 Port 8501 is not available


In [22]:
from google.colab import output
output.serve_kernel_port_as_iframe(8501)

<IPython.core.display.Javascript object>

In [23]:
!pkill -f "streamlit run app.py"

In [24]:
!pkill -f streamlit

In [25]:
from PIL import Image
img=Image.open('/mnt/data/Screenshot 2026-11-20 104241.png')
print(img.size)


FileNotFoundError: [Errno 2] No such file or directory: '/mnt/data/Screenshot 2026-11-20 104241.png'

In [26]:
import os
os.chdir("/content/project/AI-Powered Acadameic Knowledge")

!streamlit run app.py --server.port 8501 --server.address 0.0.0.0 --server.enableCORS false --server.enableXsrfProtection false



2026-11-20 05:15:27.516 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://136.64.181.163:8501

────────────────────────── Traceback (most recent call last) ───────────────────────────
  /usr/local/lib/python3.13/dist-packages/streamlit/runtime/scriptrunner/exec_code.py:  
  136 in exec_func_with_error_handling                                                  
                                                                                        
  /usr/local/lib/python3.13/dist-packages/streamlit/runtime/scriptrunner/script_runner  
  .py:936 in code_to_exec                                                               
                                                                                        
  /content/project/AI-Powered Acadameic Knowledge/app.py:7 in <module>                  
                                                

In [27]:
%pip install -r "/content/project/AI-Powered Acadameic Knowledge/requirements.txt"

In [28]:
%pip install -r "/content/project/AI-Powered Acadameic Knowledge/requirements.txt"

In [29]:
import os
os.chdir("/content/project/AI-Powered Acadameic Knowledge")

!streamlit run app.py --server.port 8501 --server.address 0.0.0.0 --server.enableCORS false --server.enableXsrfProtection false --server.headless true



2026-11-20 05:19:13.641 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://136.64.181.163:8501

  Stopping...
  Stopping...


In [30]:
from google.colab import output
output.serve_kernel_port_as_iframe(8501)

<IPython.core.display.Javascript object>

In [31]:
import requests

try:
    r = requests.get("http://127.0.0.1:8501", timeout=10)
    print("Local server status:", r.status_code)
    print(r.text[:300])
except Exception as e:
    print("Error:", e)

Error: HTTPConnectionPool(host='127.0.0.1', port=8501): Max retries exceeded with url: / (Caused by NewConnectionError("HTTPConnection(host='127.0.0.1', port=8501): Failed to establish a new connection: [Errno 111] Connection refused"))


In [32]:
import os
import subprocess
import time

os.chdir("/content/project/AI-Powered Acadameic Knowledge")

log_file = open("/content/streamlit.log", "w")

process = subprocess.Popen(
    [
        "streamlit", "run", "app.py",
        "--server.port", "8501",
        "--server.address", "0.0.0.0",
        "--server.enableCORS", "false",
        "--server.enableXsrfProtection", "false",
        "--server.headless", "true"
    ],
    stdout=log_file,
    stderr=subprocess.STDOUT
)

time.sleep(10)

print("Process status:", process.poll())
print("\n--- ERROR LOG ---")
print(open("/content/streamlit.log").read()[-5000:])

Process status: None

--- ERROR LOG ---


2026-11-20 05:21:07.550 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://136.64.181.163:8501




In [33]:
import requests

try:
    r = requests.get("http://127.0.0.1:8501", timeout=10)
    print("Server status:", r.status_code)
    print("Website response received!")
except Exception as e:
    print("Error:", e)

Server status: 200
Website response received!


In [34]:
from google.colab import output
output.serve_kernel_port_as_iframe(8501)

<IPython.core.display.Javascript object>

In [35]:
%pip install pytesseract

In [36]:
!apt-get update -qq && apt-get install -y tesseract-ocr

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tesseract-ocr is already the newest version (5.3.4-1build5).
0 upgraded, 0 newly installed, 0 to remove and 65 not upgraded.


In [37]:
import pytesseract
print("Pytesseract installed successfully!")
print("Tesseract version:", pytesseract.get_tesseract_version())

Pytesseract installed successfully!
Tesseract version: 5.3.4


In [38]:
!pkill -f streamlit

In [39]:
import os
import subprocess
import time

os.chdir("/content/project/AI-Powered Acadameic Knowledge")

log = open("/content/streamlit.log", "w")

process = subprocess.Popen(
    [
        "streamlit", "run", "app.py",
        "--server.port", "8501",
        "--server.address", "0.0.0.0",
        "--server.enableCORS", "false",
        "--server.enableXsrfProtection", "false",
        "--server.headless", "true"
    ],
    stdout=log,
    stderr=subprocess.STDOUT
)

time.sleep(10)
print(open("/content/streamlit.log").read()[-4000:])



2026-11-20 05:25:32.530 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://136.64.181.163:8501




In [40]:
import requests

try:
    r = requests.get("http://127.0.0.1:8501", timeout=10)
    print("Server status:", r.status_code)
except Exception as e:
    print("Error:", e)

Server status: 200


In [41]:
from google.colab import output
output.serve_kernel_port_as_iframe(8501)

<IPython.core.display.Javascript object>

In [42]:
%pip install pypdf

In [43]:
from pypdf import PdfReader
print("pypdf installed successfully!")

pypdf installed successfully!


In [44]:
import ast

file_path = "/content/project/AI-Powered Acadameic Knowledge/app.py"

with open(file_path, "r") as f:
    tree = ast.parse(f.read())

modules = set()

for node in ast.walk(tree):
    if isinstance(node, ast.Import):
        for name in node.names:
            modules.add(name.name.split(".")[0])
    elif isinstance(node, ast.ImportFrom) and node.module:
        modules.add(node.module.split(".")[0])

print("App imports:")
print(sorted(modules))

App imports:
['agents', 'chromadb', 'docx', 'dotenv', 'google', 'hashlib', 'os', 'pathlib', 'pdf2image', 'pypdf', 'pytesseract', 're', 'streamlit']


In [45]:
packages = {
    "chromadb": "chromadb",
    "docx": "python-docx",
    "dotenv": "python-dotenv",
    "google": "google-genai",
    "pdf2image": "pdf2image",
    "pypdf": "pypdf",
    "pytesseract": "pytesseract",
    "streamlit": "streamlit"
}

import importlib.util

for module, package in packages.items():
    status = "Installed" if importlib.util.find_spec(module) else "Missing"
    print(f"{module}: {status}")

chromadb: Installed
docx: Missing
dotenv: Installed
google: Installed
pdf2image: Missing
pypdf: Installed
pytesseract: Installed
streamlit: Installed


In [46]:
%pip install python-docx pdf2image

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [pdf2image]


In [47]:
import docx
import pdf2image

print("Both packages installed successfully!")

Both packages installed successfully!


In [48]:
import chromadb
import docx
import dotenv
import google.genai
import pdf2image
import pypdf
import pytesseract
import streamlit

print("ALL PACKAGES WORKING!")

ALL PACKAGES WORKING!


In [49]:
!pkill -f streamlit

In [50]:
import os
import subprocess
import time

os.chdir("/content/project/AI-Powered Acadameic Knowledge")

log = open("/content/streamlit.log", "w")

process = subprocess.Popen(
    [
        "streamlit", "run", "app.py",
        "--server.port", "8501",
        "--server.address", "0.0.0.0",
        "--server.enableCORS", "false",
        "--server.enableXsrfProtection", "false",
        "--server.headless", "true"
    ],
    stdout=log,
    stderr=subprocess.STDOUT
)

time.sleep(10)
print(open("/content/streamlit.log").read()[-5000:])



2026-11-20 05:30:12.782 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://136.64.181.163:8501




In [51]:
import requests

r = requests.get("http://127.0.0.1:8501", timeout=10)
print("Server status:", r.status_code)

Server status: 200


In [52]:
from google.colab import output
output.serve_kernel_port_as_iframe(8501)

<IPython.core.display.Javascript object>